### streaming-safe IQ-domain DC blocker matching the  PSD cleanup workflow.

HackRF’s receive stream is interleaved 8-bit I/Q samples, and the official documentation explicitly notes that the center spike is a DC offset artifact common to quadrature sampling systems. HackRF also recommends offset tuning first, because software DC correction can degrade content close to 0 Hz.

In [ ]:
import numpy as np
from dataclasses import dataclass


@dataclass
class HackRFDCBlockState:
    prev_x: complex = 0.0 + 0.0j
    prev_y: complex = 0.0 + 0.0j


def hackrf_interleaved_iq_to_complex64(raw_iq: np.ndarray) -> np.ndarray:
    """
    Convert HackRF interleaved signed 8-bit I/Q samples to complex64.

    Input:
        raw_iq: shape (2N,), dtype int8-like
                [I0, Q0, I1, Q1, ..., IN-1, QN-1]

    Output:
        iq: shape (N,), dtype complex64
    """
    x = np.asarray(raw_iq)
    if x.ndim != 1:
        raise ValueError("raw_iq must be a 1-D array of interleaved I/Q samples")
    if x.size % 2 != 0:
        raise ValueError("raw_iq length must be even")

    # HackRF files/buffers are 8-bit signed quadrature samples
    x = x.astype(np.int8, copy=False).astype(np.float32, copy=False)

    i = x[0::2]
    q = x[1::2]
    return (i + 1j * q).astype(np.complex64, copy=False)


def dc_block_alpha(sample_rate_hz: float, cutoff_hz: float = 30.0) -> float:
    """
    Map a small high-pass cutoff to the standard one-pole DC blocker coefficient.

    For y[n] = x[n] - x[n-1] + a*y[n-1], a should be close to 1.
    """
    fs = float(sample_rate_hz)
    fc = float(cutoff_hz)
    if fs <= 0:
        raise ValueError("sample_rate_hz must be > 0")
    if fc <= 0:
        raise ValueError("cutoff_hz must be > 0")
    a = np.exp(-2.0 * np.pi * fc / fs)
    return float(np.clip(a, 0.0, 0.999999))


def dc_block_iq_stream(
    iq: np.ndarray,
    *,
    state: HackRFDCBlockState | None = None,
    alpha: float = 0.995,
    remove_chunk_mean: bool = True,
) -> tuple[np.ndarray, HackRFDCBlockState]:
    """
    Streaming-safe complex DC blocker for HackRF IQ.

    Difference equation:
        y[n] = x[n] - x[n-1] + alpha * y[n-1]

    Parameters
    ----------
    iq : np.ndarray
        Complex input IQ, shape (N,)
    state : HackRFDCBlockState | None
        Persistent state across chunks. Pass the returned state into the next call.
    alpha : float
        Pole location, typically very close to 1.0
    remove_chunk_mean : bool
        Removes the current chunk mean before the recursive blocker.
        This helps with abrupt per-buffer bias shifts.

    Returns
    -------
    y : np.ndarray
        DC-blocked IQ, complex64
    new_state : HackRFDCBlockState
        Updated state to reuse on the next chunk
    """
    x = np.asarray(iq)
    if x.ndim != 1:
        raise ValueError("iq must be a 1-D complex array")
    if x.size == 0:
        return x.astype(np.complex64, copy=True), (state or HackRFDCBlockState())

    if not np.iscomplexobj(x):
        raise ValueError("iq must be complex-valued")
    if not (0.0 <= alpha < 1.0):
        raise ValueError("alpha must satisfy 0 <= alpha < 1")

    x = x.astype(np.complex64, copy=False)

    if remove_chunk_mean:
        # Cheap first-stage DC removal per chunk
        finite = np.isfinite(x)
        if finite.any():
            mu = x[finite].mean(dtype=np.complex64)
            x = x - mu

    s = state if state is not None else HackRFDCBlockState()

    y = np.empty_like(x, dtype=np.complex64)

    prev_x = np.complex64(s.prev_x)
    prev_y = np.complex64(s.prev_y)
    a = np.float32(alpha)

    for n in range(x.size):
        xn = x[n]
        yn = xn - prev_x + a * prev_y
        y[n] = yn
        prev_x = xn
        prev_y = yn

    new_state = HackRFDCBlockState(
        prev_x=complex(prev_x),
        prev_y=complex(prev_y),
    )
    return y, new_state


def hackrf_dc_block_from_interleaved_int8(
    raw_iq: np.ndarray,
    *,
    state: HackRFDCBlockState | None = None,
    alpha: float | None = None,
    sample_rate_hz: float | None = None,
    cutoff_hz: float = 30.0,
    remove_chunk_mean: bool = True,
    normalize_to_unit: bool = False,
) -> tuple[np.ndarray, HackRFDCBlockState]:
    """
    End-to-end helper:
      interleaved int8 IQ -> complex64 -> DC blocked IQ

    If alpha is None, it is derived from sample_rate_hz and cutoff_hz.
    """
    iq = hackrf_interleaved_iq_to_complex64(raw_iq)

    if normalize_to_unit:
        iq = iq / 128.0

    if alpha is None:
        if sample_rate_hz is None:
            raise ValueError("Provide either alpha or sample_rate_hz")
        alpha = dc_block_alpha(sample_rate_hz, cutoff_hz=cutoff_hz)

    y, new_state = dc_block_iq_stream(
        iq,
        state=state,
        alpha=alpha,
        remove_chunk_mean=remove_chunk_mean,
    )
    return y, new_state

Recommended use with HackRF One:

- Use offset tuning whenever possible, because HackRF’s own docs recommend moving the wanted signal away from 0 Hz and digitally shifting it back if needed.

- Then apply this IQ DC blocker before FFT/PSD estimation.

- Keep the cutoff very low. For most captures, cutoff_hz = 10 to 50 Hz is a good starting range; higher values remove DC faster but also eat more energy very close to 0 Hz. That tradeoff follows directly from HackRF’s warning that software DC correction may degrade signal content near DC.

Example use:

In [ ]:
# raw_bytes_or_int8: interleaved HackRF IQ from a chunk
state = HackRFDCBlockState()

alpha = dc_block_alpha(sample_rate_hz=10e6, cutoff_hz=20.0)

iq_clean, state = hackrf_dc_block_from_interleaved_int8(
    raw_bytes_or_int8,
    state=state,
    alpha=alpha,
    remove_chunk_mean=True,
    normalize_to_unit=True,
)

# Then compute PSD from iq_clean

For repeated streaming, reuse state across callback chunks. Do not reset it every block unless you intentionally want blockwise independent filtering.

Two practical notes:

First, this removes time-domain DC bias, which is the right upstream fix for the center spike. Your PSD-domain spike masking can still stay as a second safety layer for residual LO leakage or imperfect bias removal.

Second, if your wanted signal is intentionally centered exactly at DC, both the IQ blocker and the PSD center-spike remover can suppress part of it. In that case, off-center tuning is the cleaner solution.

Lastly, merge this directly into the existing HackRF capture + PSD pipeline so the DC blocker feeds the estimate_hackrf_noise_floor() function.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c4818a93-f2bf-4892-b7a2-47816cd4c47d' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>